# 02. Device Mesh and DTensor Sharding Placements

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Shoeybi et al. (Megatron-LM, 2019), Zheng et al. (Alpa, OSDI 2022), PyTorch Distributed Team (PyTorch 2.x DTensor, 2023).
- **Purpose:** Decouple global tensor mathematical semantics from physical distributed execution through topological DeviceMesh grids and automated DTensor placement specifications (`Shard`, `Replicate`, `Partial`).
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Collectives and DDP Gradient Bucketing Mechanics](01_collectives_and_ddp_gradient_bucketing_mechanics.ipynb) $\to$ `🔗 Downstream Syntheses:` [Zero Stages 1 2 3 and FSDP2 Fully Shard](03_zero_stages_1_2_3_and_fsdp2_fully_shard.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not introduce complex multi-dimensional sharding meshes when a model fits within single-GPU HBM and scales linearly with simple DDP. Sharding across non-NVLink inter-node boundaries introduces high cross-node latency penalties.

## Mathematical Foundations & Governing Equations

$$
\text{DTensor} = \langle X_{\text{global}}, \mathcal{M}_{\text{mesh}}, \{P_i\}_{i=1}^D \rangle, \quad P_i \in \{\text{Shard}(d), \text{Replicate}(), \text{Partial}(\text{Sum})\}
$$

### Asymptotic Time & Memory Complexity
- **Column-Parallel Matmul:** $\mathcal{O}(0)$ communication, $\mathcal{O}(N / M)$ parameter memory.
- **Row-Parallel Matmul:** $\mathcal{O}(2 \frac{M-1}{M} B \cdot N \beta)$ AllReduce communication, halving communication when paired with column-parallel layers.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def dtensor_shard(tensor: torch.Tensor, mesh_size: int, shard_dim: int) -> torch.Tensor:
    chunks = torch.chunk(tensor, mesh_size, dim=shard_dim)
    return torch.stack(chunks, dim=0)

def dtensor_allgather(shards: torch.Tensor, gather_dim: int) -> torch.Tensor:
    mesh_size = shards.shape[0]
    return torch.cat([shards[r] for r in range(mesh_size)], dim=gather_dim)

def dtensor_column_parallel_matmul(x: torch.Tensor, w_shards: torch.Tensor) -> torch.Tensor:
    mesh_size = w_shards.shape[0]
    return torch.stack([x @ w_shards[r] for r in range(mesh_size)], dim=0)

def dtensor_row_parallel_matmul(x_shards: torch.Tensor, w_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    mesh_size = x_shards.shape[0]
    partial_shards = torch.stack([x_shards[r] @ w_shards[r] for r in range(mesh_size)], dim=0)
    return torch.sum(partial_shards, dim=0), partial_shards

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

x_sharded = dtensor_shard(weights["global_x"], 2, 0)
assert float(torch.max(torch.abs(x_sharded - weights["x_sharded_dim0"]))) < 1e-5
x_gathered = dtensor_allgather(x_sharded, 0)
assert float(torch.max(torch.abs(x_gathered - weights["x_gathered"]))) < 1e-5

col_out = dtensor_column_parallel_matmul(weights["global_x"], weights["w_col_shards"])
assert float(torch.max(torch.abs(col_out - weights["col_out_shards"]))) < 1e-5
col_gathered = dtensor_allgather(col_out, 1)
assert float(torch.max(torch.abs(col_gathered - weights["col_out_gathered"]))) < 1e-5

row_allred, partials = dtensor_row_parallel_matmul(weights["x_row_shards"], weights["w_row_shards"])
assert float(torch.max(torch.abs(partials - weights["partial_shards"]))) < 1e-5
assert float(torch.max(torch.abs(row_allred - weights["row_out_allreduced"]))) < 1e-5
print("All DTensor Shard, AllGather, Column & Row Parallel checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: dtensor_shard(weights["global_x"], 2, 0), iters=50, name="DTensor Shard(0)"),
    benchmark_fn(lambda: dtensor_allgather(weights["x_sharded_dim0"], 0), iters=50, name="DTensor AllGather"),
    benchmark_fn(lambda: dtensor_column_parallel_matmul(weights["global_x"], weights["w_col_shards"]), iters=50, name="Column Parallel Matmul"),
    benchmark_fn(lambda: dtensor_row_parallel_matmul(weights["x_row_shards"], weights["w_row_shards"]), iters=50, name="Row Parallel Matmul"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Eager Redundant AllGather vs Row-Parallel Partial Reduction
x_shards = weights["x_row_shards"]
w_shards = weights["w_row_shards"]
full_x = weights["global_x"]
full_w = weights["w_matrix"]

# 1. Optimal Row-Parallel: compute locally on shards, single AllReduce at output
row_allred, partials = dtensor_row_parallel_matmul(x_shards, w_shards)
error_row_parallel = float(torch.norm(row_allred - (full_x @ full_w)))

# 2. Naive Eager AllGather: gathering full activations before every layer
gathered_x = dtensor_allgather(x_shards, gather_dim=1)
redundant_network_bytes = gathered_x.numel() * gathered_x.element_size()

print(f"Row-Parallel AllReduce Error: {error_row_parallel:.2e}")
print(f"Redundant Traffic Avoided:    {redundant_network_bytes} bytes")
assert error_row_parallel < 1e-5 and redundant_network_bytes > 0, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Decoupled Parallelism:** DTensor decouples mathematical operations from physical partitioning, allowing identical code to execute across 1 GPU, 8 GPUs (TP), or 1024 GPUs (3D parallel).
2. **Placement Algebra:** Transforming placements (`Shard` $\to$ `Replicate` $\to$ `Partial`) formalizes communication collective insertion as a compiler rewrite pass.
3. **Megatron Pairing:** Fusing Column-Parallel with Row-Parallel layers leaves intermediate activations sharded, halving the required collective communication count.

---
### Cross-Domain Generalisation Matrix
| Domain | Placement / Sharding Analogue | Engineering Benefit |
| :--- | :--- | :--- |
| **Large-Scale FFT / Spectral PDEs** | Pencil / Slab 2D domain decomposition | Overlaps 1D row FFT with column All-to-All transpose resharding |
| **Distributed Relational Databases** | Hash-partitioned distributed join | Joins sharded tables locally on partition keys before global aggregation |
| **Quantum Circuit Simulation** | Statevector slicing across cluster nodes | Shards $2^N$ amplitudes across distributed memory with gate-triggered swaps |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_device_mesh_and_dtensor_sharding_placements.md](../../markdown/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.md)
- **PyTorch 2.x Module:** [02_device_mesh_and_dtensor_sharding_placements.py](../../pytorch2/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.py)
- **Functional JAX Module:** [02_device_mesh_and_dtensor_sharding_placements.py](../../jax/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.py)
- **Native Rust Candle Track:** [02_device_mesh_and_dtensor_sharding_placements.rs](../../candle/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.rs)
- **Hardware Kernel (CPU / WGSL):** [02_device_mesh_and_dtensor_sharding_placements.rs](../../kernels/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.rs)
- **Golden SafeTensors:** [02_device_mesh_and_dtensor_sharding_placements.safetensors](../../weights/16_distributed_and_scalable/02_device_mesh_and_dtensor_sharding_placements.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Collectives and DDP Gradient Bucketing Mechanics](01_collectives_and_ddp_gradient_bucketing_mechanics.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 03. Zero Stages 1 2 3 and FSDP2 Fully Shard](03_zero_stages_1_2_3_and_fsdp2_fully_shard.ipynb) &rarr;